# TASK 1

Este script  servirá  para extraer metricas generales a traves de la API de MINKA. A traves del ID de un proyecto se extraeran los siguientes parámetros.


* Número de observaciones total
* Número de personas que han subido observaciones
* Número de especies registradas
* Número de personas que han contribuido con identificaciones
* Top 3 de personas con más observaciones
* Últimas 5 especies diferentes registradas (especies de las últimas observaciones, si se repite la especie, buscas la siguiente hasta sacar las últimas 5 diferentes)
* Número de observaciones mensuales desde enero 2024 (por fecha de observación, no fecha de subida, fíjate en los filtros de fecha), en formato tabla (dataframe de pandas):


In [ ]:
# Enlace principal que utilizaremos para obtener los datos de los proyectos

PROJECTS_PATH = 'https://minka-sdg.org/projects'

In [ ]:
import requests
import pandas as pd
import json
import plotly.express as px
import seaborn as sns


# El usuario inserta el id del proyecto que desea consultar y obtiene la información del proyecto en formato json.

project_id = input('Inserta el id del proyecto :')

url_id = f'{PROJECTS_PATH}/{project_id}.json'

requests.get(url_id).json()

* NÚMERO TOTAL DE OBSERVACIONES 

In [ ]:
response = requests.get(url_id).json()['project_observations_count']

print(f'El número total de observaciones del proyecto son : {response}')

* NUMERO DE PERSONAS QUE HAN SUBIDO OBSERVACIONES

In [ ]:
OBSERVERS_PATH = 'https://api.minka-sdg.org/v1/observations/observers'

url_observers = f'{OBSERVERS_PATH}?project_id={project_id}'

response = requests.get(url_observers).json()['total_results']

print(f'El número total de observadores del proyecto son : {response}')

* NÚMERO DE ESPECIES REGISTRADAS

In [ ]:
SPECIES_PATH = 'https://api.minka-sdg.org/v1/observations/species_counts'

url_species = f'{SPECIES_PATH}?project_id={project_id}'

response = requests.get(url_species).json()['total_results']

print(f'El número total de especies registradas en el proyecto son : {response}')

* NÚMERO DE PERSONAS QUE HAN CONTRIBUIDO CON IDENTIFICACIONES

In [ ]:
IDENTIFIERS_PATH = 'https://api.minka-sdg.org/v1/observations/identifiers'

url_identifiers = f'{IDENTIFIERS_PATH}?project_id={project_id}'

response = requests.get(url_identifiers).json()['total_results']

print(f'El número total de identificadores del proyecto son : {response}')

* TOP 3 PERSONAS CON MÁS OBSERVACIONES

In [ ]:
OBSERVERS_PARTH = 'https://api.minka-sdg.org/v1/observations/observers'

url_observers = f'{OBSERVERS_PATH}?project_id={project_id}'

response = requests.get(url_observers).json()['results']

print('TOP 3 PERSONAS CON MÁS OBSERVACIONES')

top_users = []
for i in range(3):
    user_data = response[i]['user']
    username = user_data['login']
    user_observations = response[i]['observation_count']
    position = ['primera', 'segunda', 'tercera'][i]

    print(f'La {position} persona con más observaciones es: {username} con {user_observations} observaciones')
    top_users.append({'username': username, 'observation_count': user_observations})

import matplotlib.pyplot as plt

plot = pd.DataFrame(top_users)
plot = plot.set_index('username')
plot.plot(kind='bar', title='TOP 3 OBSERVADORES', legend=False)
plt.show()


* ULTIMAS 5 ESPECIES REGISTRADAS

In [ ]:
OBSERVATIONS_PATH = 'https://api.minka-sdg.org/v1/observations'

url_observations = f'{OBSERVATIONS_PATH}?project_id={project_id}'

response = requests.get(url_observations).json()

# Verificar si 'results' existe en la respuesta
if 'results' in response:
    unique_species = set()  # Almacenamos los nombres cientificos de las especies que se vayan registrando
    count = 0 # Empezamos el contaje en 0 (por el principio)
    for i, result in enumerate(response['results']):  # Enumarate es bastante útil porque nos permite recorrer todos los elementos que hay en results para luego definir las variables más fácilmente
        taxon_name = result.get('taxon', {}).get('name', 'Desconocido')  # Evita errores si taxon o name no existen
        print(f"Nombre científco de la observación {i+1}: {taxon_name}") # Lo he hecho con el nombre científico porque es más específico y universal (se podria hacer que también mostrara el nombre común)

        if taxon_name not in unique_species: # Si el nombre de la especie no está en el conjunto de especies únicas, lo añadimos
            unique_species.add(taxon_name)
            count += 1
        else: 
            print(f"Ya se ha registrado esta observación {taxon_name}") # Si ya está registrado, lo indicamos
        if count == 5:
            break
else:
    print("No se encontraron resultados.")



Nombre científco de la observación 1: Hippocampus guttulatus
Nombre científco de la observación 2: Raja undulata
Nombre científco de la observación 3: Stramonita haemastoma
Nombre científco de la observación 4: Octopus vulgaris
Nombre científco de la observación 5: Sepia officinalis


*  Número de observaciones mensuales desde enero 2024

In [89]:
OBSERVATIONS_PATH = 'https://api.minka-sdg.org/v1/observations'

year = input('Inserta el año de observación que deseas consultar :')

url_observations_month = f'{OBSERVATIONS_PATH}/histogram?project_id={project_id}&year={year}&date_field=observed&interval=month_of_year'

response = requests.get(url_observations_month).json()

data = response['results']

df = pd.DataFrame(
    {"mes": [f"{year}-{int(m):02d}" for m in data["month_of_year"].keys()],
     "num_observations": list(data["month_of_year"].values())}
)

print(df)

        mes  num_observations
0   2024-01                 0
1   2024-02                 0
2   2024-03                 0
3   2024-04                 0
4   2024-05              8492
5   2024-06             11882
6   2024-07             20678
7   2024-08             30316
8   2024-09             13809
9   2024-10              7048
10  2024-11                 0
11  2024-12                 0


### MÉTRICAS ADICIONALES

* Top 5 espécies más observadas
* Top 3 localizaciones con más observaciones
* Espécies observadas en peligro de extinción/amenazadas
* Cantidad de nuevos usuarios que han hecho observaciones en el ultimo mes/semana
* Observaciones nuevas en el útlimo mes / semana

* TOP 5 ESPÉCIES MÁS OBSERVADAS

In [91]:
# Para obtener el top 5 espécies más observadas y el número total de observaciones realizamos un procedimiento similar al del TOP 3 personas con más obseravciones y el código de espécies registradas.

SPECIES_PATH = 'https://api.minka-sdg.org/v1/observations/species_counts'

url_species = f'{SPECIES_PATH}?project_id={project_id}'

response = requests.get(url_species).json()['results']

print('TOP 5 ESPÉCIES MÁS OBSERVADAS EN EL PROYECTO')

for i in range(5):
    species_name = response[i]['taxon']['name']
    count = response[i]['count']
    position = ['primera', 'segunda', 'tercera', 'cuarta', 'quinta'][i]

    print(f'La {position} espécies más observada en el proyecto es: {species_name} con {count} observaciones')


TOP 5 ESPÉCIES MÁS OBSERVADAS EN EL PROYECTO
La primera espécies más observada en el proyecto es: Diplodus vulgaris con 2203 observaciones
La segunda espécies más observada en el proyecto es: Coris julis con 2023 observaciones
La tercera espécies más observada en el proyecto es: Padina pavonica con 1768 observaciones
La cuarta espécies más observada en el proyecto es: Diplodus sargus con 1763 observaciones
La quinta espécies más observada en el proyecto es: Chromis chromis con 1728 observaciones


* TOP 3 LOCALIZACIONES CON MÁS OBSERVACIONES 

In [95]:
from collections import Counter

url_observations = f'{OBSERVATIONS_PATH}?project_id={project_id}'

#response = requests.get(url_observations).json()['results']

all_places_ids = []
page = 1
has_more_data = True

while has_more_data:
    response = requests.get(url_observations, params={'page': page, 'per_page':30}).json()
    
    if 'results' in response:
        place_ids = [item['place_ids'] for item in response['results']]
        all_places_ids.extend(place_ids)
        has_more_data = len(response['results']) == 30
    else:
        has_more_data = False
        print("No 'results' key found in the response.")
    
    page += 1

place_counts = Counter(all_places_ids)
top3_places = place_counts.most_common(3)

print('TOP 3 LUGARES CON MÁS OBSERVACIONES')

for i, (place_id, count) in enumerate(top3_places):
    print(f'Place ID:{place_id} - Repeticiones: {count}')


    

No 'results' key found in the response.


TypeError: unhashable type: 'list'